In [11]:
import pandas as pd

df = pd.read_csv('long_091826.csv')
meta = pd.read_csv('lassa_library_sequences_species_standardized_070726_ICTV_corrected.csv')

### for all species

In [32]:
pep_hits = df.groupby(['species', 'genus', 'family'])['sequence'].nunique().rename('n')
denom = meta.groupby(['species', 'genus', 'family'])['sequence'].nunique()

missing = set(pep_hits.index) - set(denom.index)
assert not missing, f"species in hits but not in meta: {sorted(missing)[:10]}"

out = pd.DataFrame({
    "n_hits": pep_hits.reindex(denom.index, fill_value=0), 
    "n_in_library": denom,
})

out["pct_hits"] = (out["n_hits"] / out["n_in_library"] * 100).round(2)

#filter coverage -- how well is represented in lib
MIN_LIB = 20
out = out[out["n_in_library"] >= MIN_LIB].sort_values("pct_hits", ascending=False)

print(f"{len(denom)} species, {len(out)} with >= {MIN_LIB} peptides in library")

441 species, 412 with >= 20 peptides in library


In [34]:
out.loc['Mammarenavirus lassaense']

,,n_hits,n_in_library,pct_hits
genus,family,,,
Mammarenavirus,Arenaviridae,76,4494,1.69


In [35]:
out.sort_values(by=['pct_hits', 'n_hits'], ascending=False).head(100)

,,,n_hits,n_in_library,pct_hits
species,genus,family,,,
S2-like reptarenavirus,Reptarenavirus,Arenaviridae,3,43,6.98
Palma virus,Phlebovirus,Phenuiviridae,2,32,6.25
Gbagroube virus,Arenavirus,Arenaviridae,2,36,5.56
Menekre virus,Arenavirus,Arenaviridae,2,44,4.55
Phlebovirus ambeense,Phlebovirus,Phenuiviridae,7,170,4.12
...,...,...,...,...,...
Orthonairovirus zirkuense,Orthonairovirus,Nairoviridae,5,246,2.03
Mammarenavirus ganziense,Mammarenavirus,Arenaviridae,4,198,2.02
Phlebovirus candiruense,Phlebovirus,Phenuiviridae,10,499,2.00


In [43]:
out.groupby('genus')['pct_hits'].head(3).reset_index().sort_values(by=['genus', 'pct_hits'], ascending=[True,False]).head(50)

,species,genus,family,pct_hits
41,Actinovirus triacanthodis,Actinovirus,Hantaviridae,1.91
50,Actinovirus bernense,Actinovirus,Hantaviridae,1.75
78,Actinovirus halieutaeae,Actinovirus,Hantaviridae,0.00
67,Agnathovirus eptatreti,Agnathovirus,Hantaviridae,1.17
72,Alphavirus chikungunya,Alphavirus,Togaviridae,0.59
26,Antennavirus hirsutum,Antennavirus,Arenaviridae,2.72
28,Antennavirus striale,Antennavirus,Arenaviridae,2.53
69,Antennavirus salmonis,Antennavirus,Arenaviridae,1.02
2,Gbagroube virus,Arenavirus,Arenaviridae,5.56
3,Menekre virus,Arenavirus,Arenaviridae,4.55


In [ ]:
pep_hits_norm.sort_values(ascending=False).head(30)

species
Phlebovirus sp. PAN 483391                      10.00
Amga virus                                      10.00
Mopeia Lassa virus reassortant 29                8.33
Rodent Mammarenavirus                            7.69
S2-like reptarenavirus                           6.98
Muraenolepis arenavirus                          6.90
Palma virus                                      6.25
Gbagroube virus                                  5.56
Arenavirus H0030026                              4.55
Menekre virus                                    4.55
Phlebovirus ambeense                             4.12
Hartmanivirus quadrati                           3.94
Orthohantavirus mamorense                        3.90
Uukuvirus dabieshanense                          3.88
Mobatvirus quezonense                            3.80
Hemidactylus frenatus arenavirus                 3.74
Sclerotinia sclerotiorum phlebo-like virus 1     3.67
Ixovirus gakugsaense                             3.60
Phlebovirus riogrand

### more LASV

In [ ]:
lasv = df.loc[df['species'] == 'Mammarenavirus lassaense']

meta_lasv = meta.loc[meta['species'] == 'Mammarenavirus lassaense']

In [ ]:
lasv_protein_std = lasv.groupby('protein_std')['peptide'].nunique()
lasv_protein_std 

protein_std
glycoprotein      5
nucleoprotein     7
polymerase       62
z protein         2
Name: peptide, dtype: int64

In [ ]:
meta_group = meta_lasv.groupby('protein_std')['seq_id'].size()

In [ ]:
meta_group

protein_std
glycoprotein      228
nucleoprotein     255
polymerase       3696
z protein         315
Name: seq_id, dtype: int64

In [ ]:
for protein in meta_group.keys():
    print(f'total peptides belonging to {protein}', meta_group[protein])
    total_peps = meta_group[protein]
    react_peps = lasv_protein_std[protein]
    print(f'total reac peps for {protein}', react_peps)

    frac = react_peps / total_peps
    frac = round(frac * 100, 1)

    print(f'% of reactive peps for {protein}', frac)


total peptides belonging to glycoprotein 228
total reac peps for glycoprotein 5
% of reactive peps for glycoprotein 2.2
total peptides belonging to nucleoprotein 255
total reac peps for nucleoprotein 7
% of reactive peps for nucleoprotein 2.7
total peptides belonging to polymerase 3696
total reac peps for polymerase 62
% of reactive peps for polymerase 1.7
total peptides belonging to z protein 315
total reac peps for z protein 2
% of reactive peps for z protein 0.6
